# exp-05 — the shared step first, by design

Reads exp-05's runs under `experiments/runs/exp-05-shared-step-first/` and
exp-04's references under `experiments/runs/exp-04-panel-major/`, scores every
leaf not scored yet, and compares `[A|B, {C_i}] ← D` — D invokes
`classify-panels` once, then each check — with exp-04's `A|B ← C_i ← D` of the
same contract shape.

**This notebook spends nothing.** The sessions are produced by

```bash
python experiments/exp-05-shared-step-first/run.py
```

The claims, margins and statistics are fixed in the preregistration,
[`thinking/experiments/exp-05-shared-step-first.md`](../../thinking/experiments/exp-05-shared-step-first.md).
This notebook computes the numbers those rules read; it does not decide.

## The conditions

| condition | checklist | entry | arrangement | runs |
|---|---|---|---|---|
| `CM exp-05` | `fig-checklist-exp05` | `do-fig-checklist-cm` v2 | `[A\|B, {C_i}] ← D` | `exp-05…/cm/pinned` |
| `PM exp-05` | `fig-checklist-exp05` | `do-fig-checklist-pm` v2 | `[A\|B, {C_i}] ← D` | `exp-05…/pm/pinned` |
| `CM exp-04` | `fig-checklist-exp04` | `do-fig-checklist-cm` v1 | `A\|B ← C_i ← D` | `exp-04…/cm/<checks>@v3` — reused |
| `PM exp-04` | `fig-checklist-exp04` | `do-fig-checklist-pm` v1 | `A\|B ← C_i ← D` | `exp-04…/pm/<checks>@v3` — reused |
| `PM exp-04 sanity` | `fig-checklist-exp04` | `do-fig-checklist-pm` v1 | `A\|B ← C_i ← D` | `exp-05…/sanity/pm/<checks>@v3`, one replicate |

The contracts are byte-identical across the two checklists, so one set of code
reads all five. Both shapes are rewritten into per-check terms as in exp-04 —
PM's shared `panel_label` and row set counted towards every check.

## The gold

**`gold-v4`**, as exp-04. This cell pins the scorer to it and must run first.

In [ ]:
from soda_mmqc.gold import pin_gold

GOLD_TAG = "gold-v4"
pin_gold(GOLD_TAG)

In [ ]:
import importlib.util
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc.config import gold_root
from soda_mmqc.core.run_layout import iter_leaves
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check
from soda_mmqc.reporting import (
    arm_contrast,
    layer1_counts_by_example,
    layer2_counts,
    layer_s_counts_by_example,
    load_run_root,
    non_response_counts,
    plot_stacked_counts,
    rate_contrast,
    scores_frame,
)
from soda_mmqc.reporting.styles import (
    LAYER1_ORDER,
    LAYER2_BINARY_ORDER,
    LAYER2_GRADED_ORDER,
    LAYER_S_ORDER,
)


def _load(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


exp05 = _load("exp05_run", Path("../../experiments/exp-05-shared-step-first/run.py"))
panels = _load("exp04_panels", Path("../../experiments/exp-04-panel-major/panels.py"))
assert exp05.GOLD_TAG == GOLD_TAG, "the runner and this notebook name different gold"

CHECKS, ENTRIES = exp05.CHECKS, exp05.ENTRIES
CLASSIFY, IDENTIFY = exp05.CLASSIFY, exp05.IDENTIFY
REFERENCE_ARM = exp05.SANITY_ARM   # exp-04's A|B <- C_i <- D

#: exp-05's runs; set EXP05_ROOT to `.../smoke` to exercise the notebook.
RUNS05 = Path(os.environ.get("EXP05_ROOT", "../../experiments/runs/exp-05-shared-step-first"))
RUNS04 = Path("../../experiments/runs/exp-04-panel-major")

GOLD = gold_root()
DELTA = {"layer S": 0.0125, "layer 1": 0.02, "layer 2": 0.02}
#: Pre-declared as expected to be uninformative.
UNINFORMATIVE = {("layer 2", "micrograph-scale-bar", "outputs[].from_the_image")}
SEED = 20261007
RESAMPLES = 10_000

#: (condition, shape, checklist, entry, root, arm) for every run.
RUN_ROOTS = (
    ("CM exp-05", "cm", exp05.CHECKLIST, ENTRIES["cm"], RUNS05 / "cm", "pinned"),
    ("PM exp-05", "pm", exp05.CHECKLIST, ENTRIES["pm"], RUNS05 / "pm", "pinned"),
    ("CM exp-04", "cm", "fig-checklist-exp04", ENTRIES["cm"], RUNS04 / "cm", REFERENCE_ARM),
    ("PM exp-04", "pm", "fig-checklist-exp04", ENTRIES["pm"], RUNS04 / "pm", REFERENCE_ARM),
    ("PM exp-04 sanity", "pm", "fig-checklist-exp04", ENTRIES["pm"], RUNS05 / "sanity" / "pm",
     REFERENCE_ARM),
)
CONDITION_ORDER = tuple(r[0] for r in RUN_ROOTS)
#: The hypotheses' comparisons: exp-05 (variant) against exp-04 (baseline).
PAIRS = {"CM exp-05": "CM exp-04", "PM exp-05": "PM exp-04"}
#: Hypothesis 1's six comparisons are read on five replicates each side; the
#: sanity rerun is a look, never part of a gate.
GATED = ("CM exp-05", "CM exp-04", "PM exp-05", "PM exp-04")


def _cm_owner(value: str) -> str | None:
    for check in CHECKS:
        if value == check or value.startswith((f"{check}[", f"{check}.")):
            return check
    return None


def to_per_check(frame: pd.DataFrame, shape: str, cond: str) -> pd.DataFrame:
    """exp-04's rewrite into per-check terms, labelled with one condition."""
    frame = frame.copy()
    frame["condition"] = cond
    key = "property" if "property" in frame else "list_key"
    if shape == "cm":
        frame["check"] = [_cm_owner(v) for v in frame[key]]
        if frame["check"].isna().any():
            raise ValueError(f"CM {key}s owned by no check: "
                             f"{sorted(frame.loc[frame['check'].isna(), key].unique())}")
        for column in ("property", "list_key"):
            if column in frame:
                frame[column] = ["outputs" + v[len(c):] for v, c in zip(frame[column], frame["check"])]
        return frame
    parts = []
    for check in CHECKS:
        if key == "list_key":
            part = frame[frame["list_key"] == "outputs"].copy()
        else:
            own = frame[key].str.startswith(f"outputs[].{check}.")
            shared = frame[key] == "outputs[].panel_label"
            part = frame[own | shared].copy()
            part[key] = part[key].str.replace(f"outputs[].{check}.", "outputs[].", regex=False)
        part["check"] = check
        parts.append(part)
    if key == "property":
        owned = {p for p in frame[key]
                 if p == "outputs[].panel_label" or any(p.startswith(f"outputs[].{c}.") for c in CHECKS)}
        if set(frame[key]) - owned:
            raise ValueError(f"PM properties owned by no check: {sorted(set(frame[key]) - owned)}")
    return pd.concat(parts, ignore_index=True)


def leaves(root: Path, arm: str):
    if not root.is_dir():
        return
    for a, replicate, leaf in iter_leaves(root):
        if a == arm:
            yield replicate, leaf


for cond, shape, checklist, entry, root, arm in RUN_ROOTS:
    print(f"{cond:<18} {root.resolve() / arm}")

## Score every leaf that is not scored yet

exp-04's reference leaves are already scored by exp-04's notebook against the
same gold; exp-05's are scored here. Stale analyses are refused, as before.

In [ ]:
stale = []
for cond, shape, checklist, entry, root, arm in RUN_ROOTS:
    for replicate, leaf in leaves(root, arm):
        analysis = leaf / ANALYSIS_FILENAME
        if not analysis.is_file():
            score_check(checklist, entry, leaf, save=True)
            print(f"scored {cond} rep-{replicate:02d}")
            continue
        predictions = list(leaf.rglob("prediction.json"))
        gold = [GOLD / p.parent.relative_to(leaf) / "checks" / entry / "expected_output.json"
                for p in predictions]
        newest = max((p.stat().st_mtime for p in predictions + gold if p.is_file()), default=0.0)
        if newest > analysis.stat().st_mtime:
            stale.append(leaf)
if stale:
    raise RuntimeError("predictions or gold newer than their analysis in:\n  "
                       + "\n  ".join(map(str, stale)))
print("all leaves scored, none stale")

## Endpoint 0 — dispatch, and the shared step first

From each session's `skill_trace.json`:

- **all dispatched** — all three check skills invoked;
- **no check** — the session answered without invoking any check skill;
- **classify first** — `classify-panels` invoked before any check, the designed
  order in exp-05; and how many times;
- **spontaneous** — any skill outside the condition's closure.

In [ ]:
def read_trace(path: Path) -> list[str]:
    try:
        entries = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return []
    return [e.get("skill") for e in entries if isinstance(e, dict) and e.get("skill")]


def read_answer(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


rows = []
for cond, shape, checklist, entry, root, arm in RUN_ROOTS:
    expected = {entry, CLASSIFY, *CHECKS}
    for replicate, leaf in leaves(root, arm):
        for trace in leaf.rglob("intermediates/skill_trace.json"):
            skills = read_trace(trace)
            called = [s for s in skills if s in CHECKS]
            first_check = skills.index(called[0]) if called else len(skills)
            rows.append({
                "condition": cond,
                "replicate": replicate,
                "example": str(trace.parent.parent.relative_to(leaf)),
                "all_dispatched": set(CHECKS) <= set(called),
                "no_check": not called,
                "classify_calls": skills.count(CLASSIFY),
                "classify_first": CLASSIFY in skills[:first_check],
                "order": " > ".join(dict.fromkeys(called)),
                "spontaneous": tuple(sorted(set(skills) - expected)),
            })
sessions = pd.DataFrame(rows)
summary0 = sessions.groupby("condition").agg(
    sessions=("example", "size"),
    all_dispatched=("all_dispatched", "sum"),
    no_check=("no_check", "sum"),
    classify_first=("classify_first", "sum"),
).reindex(CONDITION_ORDER)
summary0

In [ ]:
print(sessions.groupby(["condition", "classify_calls"]).size().unstack(fill_value=0).reindex(CONDITION_ORDER))
print()
print(sessions.groupby(["condition", "order"]).size().rename("sessions").reset_index().to_string())
print()
spont = sessions.explode("spontaneous").dropna(subset=["spontaneous"])
print("spontaneous skills:", "none" if spont.empty else spont.groupby(["condition", "spontaneous"]).size())

## One row per property, per replicate, per example

In [ ]:
frames, layer_s, layer_1, layer_2, non_response = [], [], [], [], []
for cond, shape, checklist, entry, root, arm in RUN_ROOTS:
    if not root.is_dir():
        continue
    runs = load_run_root(root, checklist=checklist, check=entry)
    for out, fn in ((frames, scores_frame), (layer_s, layer_s_counts_by_example),
                    (layer_1, layer1_counts_by_example), (layer_2, layer2_counts),
                    (non_response, non_response_counts)):
        f = fn(runs)
        f = f[f["arm"] == arm]
        if len(f):
            out.append(to_per_check(f, shape, cond))

scores = pd.concat(frames, ignore_index=True)
layer_s = pd.concat(layer_s, ignore_index=True)
layer_1 = pd.concat(layer_1, ignore_index=True)
layer_2 = pd.concat(layer_2, ignore_index=True)
non_response = pd.concat(non_response, ignore_index=True)

props = scores.groupby(["check", "condition"])["property"].agg(frozenset)
for check in CHECKS:
    sets = {c: props[check][c] for c in props[check].index}
    if len(set(sets.values())) > 1:
        raise ValueError(f"{check}: conditions score different properties: {sets}")
print(f"{len(scores)} score rows; replicates per condition:")
scores.groupby("condition")["replicate"].nunique().reindex(CONDITION_ORDER)

## Non-response and failures

In [ ]:
print(non_response.groupby(["check", "condition"])["empty"].sum().unstack(fill_value=0)
      .reindex(columns=CONDITION_ORDER))


def benchmark(checklist, entry):
    path = Path("../../soda_mmqc/data/checklist") / checklist / entry / "benchmark.json"
    return json.loads(path.read_text(encoding="utf-8"))["examples"]


missing = []
for cond, shape, checklist, entry, root, arm in RUN_ROOTS:
    wanted = benchmark(checklist, entry)
    for replicate, leaf in leaves(root, arm):
        missing.append({"condition": cond, "replicate": replicate,
                        "without_prediction": sum(not (leaf / ex / "prediction.json").is_file()
                                                  for ex in wanted),
                        "sessions": len(wanted)})
pd.DataFrame(missing).groupby("condition")[["without_prediction", "sessions"]].sum().reindex(CONDITION_ORDER)

## The outcomes, condition beside condition

In [ ]:
plot_stacked_counts(
    layer_s, group="check", category="outcome", series="condition",
    order=LAYER_S_ORDER, series_order=CONDITION_ORDER, ylabel="rows",
    title="Layer S row outcomes, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_1, group="check", category="layer1", series="condition",
    order=LAYER1_ORDER, series_order=CONDITION_ORDER, ylabel="instances",
    title="Layer 1 applicability calls, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2['layer2'].isin(LAYER2_BINARY_ORDER)], group="check", category="layer2", series="condition",
    order=LAYER2_BINARY_ORDER, series_order=CONDITION_ORDER, ylabel="instances",
    title="Layer 2, binary properties, summed over examples and replicates",
)

In [ ]:
plot_stacked_counts(
    layer_2[layer_2['layer2'].isin(LAYER2_GRADED_ORDER)], group="check", category="layer2", series="condition",
    order=LAYER2_GRADED_ORDER, series_order=CONDITION_ORDER, ylabel="instances",
    title="Layer 2, graded and multiclass properties, summed over examples and replicates",
)

## The verdict rule

exp-05 against exp-04 of the same shape, paired by example, bootstrapped over
the 38 examples. **Non-inferior** if the CI's lower bound is above −δ,
**degraded** if the CI lies below −δ, **inconclusive** otherwise;
**uninformative** only for the pre-declared cell, when its half-width exceeds δ.
Fixed sequence S → 1 → 2.

In [ ]:
def verdict(low, high, delta):
    if pd.isna(low) or pd.isna(high):
        return "no data"
    if low > -delta:
        return "non-inferior"
    if high < -delta:
        return "degraded"
    return "inconclusive"


def mark_uninformative(frame, endpoint, delta):
    frame = frame.copy()
    prop = frame["property"] if "property" in frame else [None] * len(frame)
    frame["pre_declared"] = [(endpoint, c, p) in UNINFORMATIVE for c, p in zip(frame["check"], prop)]
    frame.loc[frame["pre_declared"] & (frame["half_width"] > delta), "verdict"] = "uninformative"
    return frame


def rate_gate(frame, *, category, numerator, denominator, group, delta, label):
    out = []
    for check in CHECKS:
        sub = frame[frame["check"] == check]
        for variant, baseline in PAIRS.items():
            pair = sub[sub["condition"].isin((variant, baseline))]
            if pair["condition"].nunique() < 2:
                continue
            c = rate_contrast(
                pair.drop(columns=["arm"]).rename(columns={"condition": "arm"}),
                baseline=baseline, variant=variant, category=category, numerator=numerator,
                denominator=denominator, group=group, resamples=RESAMPLES, seed=SEED)
            c.insert(0, "shape", variant[:2])
            c.insert(0, "endpoint", label)
            out.append(c)
    if not out:
        return pd.DataFrame()
    r = pd.concat(out, ignore_index=True)
    r["half_width"] = (r["ci_high"] - r["ci_low"]) / 2
    r["verdict"] = [verdict(lo, hi, delta) for lo, hi in zip(r["ci_low"], r["ci_high"])]
    return mark_uninformative(r, label, delta)


GATE_COLUMNS = ["shape", "check", "difference", "ci_low", "ci_high", "half_width",
                "n_examples", "paired_fraction", "pre_declared", "verdict"]

## Gate 1 — layer S

`correct_row / (correct_row + missing_row)` per check.

In [ ]:
gate1 = rate_gate(layer_s, category="outcome", numerator=("correct_row",),
                  denominator=("correct_row", "missing_row"), group=("check", "list_key"),
                  delta=DELTA["layer S"], label="layer S")
gate1[GATE_COLUMNS] if not gate1.empty else gate1

### Layer S, reported with spurious rows; spurious and missing panels per figure

Reported, not gated.

In [ ]:
def per_example_rate(frame, category, numerator, denominator=None):
    counts = (frame.groupby(["condition", "check", "example", category])["count"].sum()
              .unstack(fill_value=0))
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den = counts[[c for c in (denominator or counts.columns) if c in counts]].sum(axis=1)
    return (num / den.where(den > 0)).rename("rate").reset_index()


rates = {}
for label, den in (("found / (correct + missing)", ("correct_row", "missing_row")),
                   ("correct / (correct + missing + spurious)",
                    ("correct_row", "missing_row", "spurious_row"))):
    r = per_example_rate(layer_s, "outcome", ("correct_row",), den)
    rates[label] = r.groupby(["check", "condition"])["rate"].mean()
print(pd.DataFrame(rates).unstack("check").reindex(CONDITION_ORDER))

rows = []
for cond, shape, checklist, entry, root, arm in RUN_ROOTS:
    for replicate, leaf in leaves(root, arm):
        for prediction in leaf.rglob("prediction.json"):
            example = str(prediction.parent.relative_to(leaf))
            gold = json.loads((GOLD / example / "checks" / ENTRIES["pm"] / "expected_output.json")
                              .read_text(encoding="utf-8"))
            rows.append({"condition": cond, "replicate": replicate, "example": example,
                         **panels.panel_counts([r["panel_label"] for r in gold["outputs"]],
                                               read_answer(prediction), shape)})
panel_rows = pd.DataFrame(rows)
totals = panel_rows.groupby("condition")[["spurious_panels", "missing_panels",
                                         "partly_missing_panels"]].sum()
totals["sessions"] = panel_rows.groupby("condition").size()
totals.reindex(CONDITION_ORDER)

## Gate 2 — layer 1

In [ ]:
gate2 = rate_gate(layer_1, category="layer1", numerator=("correct_applicable", "correct_NA"),
                  denominator=None, group=("check",), delta=DELTA["layer 1"], label="layer 1")
gate2[GATE_COLUMNS] if not gate2.empty else gate2

## Gate 3 — layer 2

`mean_score` per property, paired by example, replicates averaged first;
`difference ± 1.96·se`.

In [ ]:
rows = []
for check in CHECKS:
    sub = scores[scores["check"] == check]
    for variant, baseline in PAIRS.items():
        pair = sub[sub["condition"].isin((variant, baseline))]
        if pair["condition"].nunique() < 2:
            continue
        c = arm_contrast(pair.drop(columns=["arm"]).rename(columns={"condition": "arm"}),
                         baseline=baseline, variant=variant)
        c.insert(0, "shape", variant[:2])
        rows.append(c)
gate3 = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if not gate3.empty:
    gate3["half_width"] = 1.96 * gate3["se"].astype(float)
    gate3["ci_low"] = gate3["difference"].astype(float) - gate3["half_width"]
    gate3["ci_high"] = gate3["difference"].astype(float) + gate3["half_width"]
    gate3["verdict"] = [verdict(lo, hi, DELTA["layer 2"]) for lo, hi in zip(gate3["ci_low"], gate3["ci_high"])]
    gate3 = mark_uninformative(gate3, "layer 2", DELTA["layer 2"])
    gate3 = gate3.sort_values("difference")[["shape", "check", "property", "difference", "ci_low",
                                             "ci_high", "half_width", "n_examples",
                                             "paired_fraction", "pre_declared", "verdict"]]
gate3

## Through the fixed sequence — hypothesis 1

In [ ]:
def gate_state(verdicts):
    v = set(verdicts) - {"uninformative"}
    if not v:
        return "uninformative"
    if "degraded" in v:
        return "degraded"
    return "non-inferior" if v <= {"non-inferior"} else "inconclusive"


rows = []
for variant in PAIRS:
    for check in CHECKS:
        states = {}
        for label, gate in (("layer S", gate1), ("layer 1", gate2), ("layer 2", gate3)):
            sub = gate[(gate["shape"] == variant[:2]) & (gate["check"] == check)] if not gate.empty else gate
            states[label] = gate_state(sub["verdict"]) if len(sub) else "no data"
        stop = next((g for g, s in states.items() if s != "non-inferior"), None)
        rows.append({"shape": variant[:2], "check": check, **states,
                     "stops_at": stop or "non-inferior at all three"})
pd.DataFrame(rows)

## Hypothesis 2 — PM output tokens, and the other costs

Per figure, averaged over replicates; exp-05 ÷ exp-04 of the same shape, mean of
per-figure ratios, percentile bootstrap. Hypothesis 2 is the **PM output
tokens** row: **fewer** if the interval lies below 1. Everything else is
reported without a direction.

In [ ]:
def read_usage(path: Path) -> dict:
    usage = json.loads(path.read_text(encoding="utf-8")).get("usage") or {}
    tokens = usage.get("usage") or {}
    return {
        "cost": usage.get("total_cost_usd"),
        "turns": usage.get("num_turns"),
        "seconds": (usage.get("duration_ms") or 0) / 1000,
        "input_tokens": sum(tokens.get(k) or 0 for k in
                            ("input_tokens", "cache_creation_input_tokens", "cache_read_input_tokens")),
        "output_tokens": tokens.get("output_tokens") or 0,
    }


rows = []
for cond, shape, checklist, entry, root, arm in RUN_ROOTS:
    for replicate, leaf in leaves(root, arm):
        for audit in leaf.rglob("intermediates/tool_audit.json"):
            rows.append({"condition": cond, "replicate": replicate,
                         "example": str(audit.parent.parent.relative_to(leaf)), **read_usage(audit)})
usage = pd.DataFrame(rows)
QUANTITIES = ("output_tokens", "cost", "input_tokens", "turns", "seconds")
per_figure = usage.groupby(["condition", "example"])[list(QUANTITIES)].mean()
print(per_figure.groupby(level=0).mean().reindex(CONDITION_ORDER))


def ratio_rows(variant, baseline, rng):
    out = []
    both = per_figure.loc[variant].join(per_figure.loc[baseline], lsuffix="_v", rsuffix="_b",
                                        how="inner")
    for q in QUANTITIES:
        r = (both[f"{q}_v"] / both[f"{q}_b"]).replace([np.inf, -np.inf], np.nan).dropna().to_numpy()
        boots = rng.choice(r, size=(RESAMPLES, len(r)), replace=True).mean(axis=1)
        low, high = np.percentile(boots, 2.5), np.percentile(boots, 97.5)
        out.append({"comparison": f"{variant} / {baseline}", "quantity": q, "ratio": r.mean(),
                    "ci_low": low, "ci_high": high, "n_figures": len(r),
                    "verdict": ("fewer" if high < 1 else "more" if low > 1 else "inconclusive")
                    if (variant, q) == ("PM exp-05", "output_tokens") else "reported"})
    return out


rng = np.random.default_rng(SEED)
present = set(per_figure.index.get_level_values(0))
rows = [r for v, b in PAIRS.items() if {v, b} <= present for r in ratio_rows(v, b, rng)]
pd.DataFrame(rows)

## The sanity rerun

exp-04's PM `A|B ← C_i ← D`, rerun once on every figure, beside exp-04's five
replicates. **Not a gate**: one replicate against five is a look, not a test.

In [ ]:
sanity = []
if {"PM exp-04 sanity", "PM exp-04"} <= present:
    rng = np.random.default_rng(SEED)
    sanity += ratio_rows("PM exp-04 sanity", "PM exp-04", rng)
look = pd.DataFrame(sanity)
for label, frame, category, numerator, den in (
        ("layer S", layer_s, "outcome", ("correct_row",), ("correct_row", "missing_row")),
        ("layer 1", layer_1, "layer1", ("correct_applicable", "correct_NA"), None)):
    r = per_example_rate(frame, category, numerator, den)
    print(label, r[r["condition"].isin(("PM exp-04", "PM exp-04 sanity"))]
          .groupby(["check", "condition"])["rate"].mean().unstack())
look

## Replicate variance, for later margins

In [ ]:
def replicate_variance(frame, category, numerator, denominator=None):
    counts = (frame[frame["condition"].isin(GATED)]
              .groupby(["condition", "check", "example", "replicate", category])["count"].sum()
              .unstack(fill_value=0))
    num = counts[[c for c in numerator if c in counts]].sum(axis=1)
    den = counts[[c for c in (denominator or counts.columns) if c in counts]].sum(axis=1)
    rate = (num / den.where(den > 0)).rename("rate").reset_index()
    var = rate.groupby(["condition", "check", "example"])["rate"].var(ddof=1).rename("var").reset_index()
    out = var.groupby(["condition", "check"]).agg(mean_var=("var", "mean"), n_examples=("var", "count"))
    out["planned_half_width_5v5"] = 1.96 * np.sqrt(2 * out["mean_var"] / 5 / out["n_examples"])
    return out


pd.concat({
    "layer S": replicate_variance(layer_s, "outcome", ("correct_row",), ("correct_row", "missing_row")),
    "layer 1": replicate_variance(layer_1, "layer1", ("correct_applicable", "correct_NA")),
}, names=["endpoint"])

## What this does not do

It decides nothing; the reading of these numbers is fixed in
[`thinking/experiments/exp-05-shared-step-first.md`](../../thinking/experiments/exp-05-shared-step-first.md).